# Anomaly Detection from Open Data

## Notebook objective

This notebook uses the NHS 111 Minimum Data Set time series file to:

1. load and clean the raw Excel data  
2. create service level features for anomaly detection  
3. generate a synthetic labelled dataset based on real NHS patterns  
4. run anomaly detection using Isolation Forest  
5. visualise unusual operational patterns  
6. summarise key findings and takeaways  

> This is **service level operational anomaly detection**. It is not patient level misuse detection because the source data is aggregated.

## Dataset used

**Source file in this notebook**

`20210708-NHS-111-MDS-time-series-to-March-2021.xlsx`

**Sheet used**

`Raw`

The workbook is a report style Excel file, so the first step is to reconstruct usable column names from the multi row header.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Load the workbook and inspect the available sheets

In [ ]:
file_path = "20210708-NHS-111-MDS-time-series-to-March-2021.xlsx"

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

## 2. Read the Raw sheet

The Raw sheet contains the usable time series data, but the header is split across two rows.

In [ ]:
raw_df = pd.read_excel(file_path, sheet_name="Raw", header=None)

print("Raw shape:", raw_df.shape)
raw_df.head(10)

## 3. Rebuild the column names from the two header rows

In [ ]:
top_header = raw_df.iloc[4]
bottom_header = raw_df.iloc[5]

final_columns = []

for top, bottom in zip(top_header, bottom_header):
    if pd.isna(top) and pd.isna(bottom):
        final_columns.append(None)
    elif pd.isna(bottom):
        final_columns.append(str(top).strip())
    elif pd.isna(top):
        final_columns.append(str(bottom).strip())
    else:
        final_columns.append(f"{str(top).strip()}_{str(bottom).strip()}")

df = raw_df.iloc[6:].copy()
df.columns = final_columns
df = df.reset_index(drop=True)

# remove fully empty columns
df = df.loc[:, df.columns.notna()]

# clean column names
df.columns = (
    pd.Index(df.columns)
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("%", "percent", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("&", "and", regex=False)
)

print("Number of columns:", len(df.columns))
df.columns.tolist()[:20]

## 4. Select the columns needed for operational anomaly detection

We will use these fields:

- region  
- provider  
- period  
- code  
- area  
- population  
- total calls offered  
- calls abandoned  
- calls answered  

These are enough to build a strong service level anomaly detection example.

In [ ]:
selected_columns = [
    "region",
    "provider",
    "period",
    "code",
    "area",
    "4.3_population",
    "5.3_total_calls_offered",
    "5.6_no_of_abandoned_calls",
    "5.7_no_calls_answered"
]

clean_df = df[selected_columns].copy()

clean_df = clean_df.rename(columns={
    "4.3_population": "population",
    "5.3_total_calls_offered": "calls_offered",
    "5.6_no_of_abandoned_calls": "calls_abandoned",
    "5.7_no_calls_answered": "calls_answered"
})

clean_df.head()

## 5. Clean data types and remove unusable rows

In [ ]:
numeric_columns = ["population", "calls_offered", "calls_abandoned", "calls_answered"]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df["period"] = pd.to_datetime(clean_df["period"], errors="coerce")

clean_df = clean_df.dropna(subset=["calls_offered", "calls_answered", "calls_abandoned"]).copy()

print("Clean dataset shape:", clean_df.shape)
clean_df.info()

## 6. Feature engineering

We now create simple but meaningful operational features:

- `unanswered_calls`
- `answer_rate`
- `abandon_rate`
- `calls_per_1000_population`

In [ ]:
clean_df["unanswered_calls"] = clean_df["calls_offered"] - clean_df["calls_answered"]

clean_df["answer_rate"] = np.where(
    clean_df["calls_offered"] > 0,
    clean_df["calls_answered"] / clean_df["calls_offered"],
    0
)

clean_df["abandon_rate"] = np.where(
    clean_df["calls_offered"] > 0,
    clean_df["calls_abandoned"] / clean_df["calls_offered"],
    0
)

clean_df["calls_per_1000_population"] = np.where(
    clean_df["population"] > 0,
    (clean_df["calls_offered"] / clean_df["population"]) * 1000,
    0
)

clean_df.head()

## 7. Quick data quality checks

In [ ]:
clean_df.describe(include="all")

## 8. Save the cleaned real dataset

In [ ]:
clean_output_path = "nhs_111_clean_dataset.csv"
clean_df.to_csv(clean_output_path, index=False)
clean_output_path

## 9. Create a synthetic labelled anomaly dataset

The real NHS source is aggregated and does not provide anomaly labels.

So we create a synthetic training dataset by:

1. starting from real NHS service patterns  
2. adding small realistic variation  
3. injecting a small number of abnormal records  
4. creating a label called `is_anomaly`  

This is a standard prototype approach for industry analytics when labels are not available.

In [ ]:
np.random.seed(42)

synthetic_df = clean_df.copy()

# add realistic variation
synthetic_df["calls_offered"] = synthetic_df["calls_offered"] * np.random.uniform(0.90, 1.10, len(synthetic_df))
synthetic_df["calls_answered"] = synthetic_df["calls_answered"] * np.random.uniform(0.90, 1.10, len(synthetic_df))
synthetic_df["calls_abandoned"] = synthetic_df["calls_abandoned"] * np.random.uniform(0.90, 1.20, len(synthetic_df))

# keep values logically valid
synthetic_df["calls_answered"] = np.minimum(synthetic_df["calls_answered"], synthetic_df["calls_offered"])
synthetic_df["calls_abandoned"] = np.minimum(synthetic_df["calls_abandoned"], synthetic_df["calls_offered"])

# recalculate features
synthetic_df["unanswered_calls"] = synthetic_df["calls_offered"] - synthetic_df["calls_answered"]
synthetic_df["answer_rate"] = np.where(
    synthetic_df["calls_offered"] > 0,
    synthetic_df["calls_answered"] / synthetic_df["calls_offered"],
    0
)
synthetic_df["abandon_rate"] = np.where(
    synthetic_df["calls_offered"] > 0,
    synthetic_df["calls_abandoned"] / synthetic_df["calls_offered"],
    0
)
synthetic_df["calls_per_1000_population"] = np.where(
    synthetic_df["population"] > 0,
    (synthetic_df["calls_offered"] / synthetic_df["population"]) * 1000,
    0
)

synthetic_df["is_anomaly"] = 0

anomaly_count = max(1, int(len(synthetic_df) * 0.05))
anomaly_indices = np.random.choice(synthetic_df.index, size=anomaly_count, replace=False)

# inject abnormal patterns
synthetic_df.loc[anomaly_indices, "calls_offered"] = (
    synthetic_df.loc[anomaly_indices, "calls_offered"] * np.random.uniform(1.8, 3.0, anomaly_count)
)
synthetic_df.loc[anomaly_indices, "calls_abandoned"] = (
    synthetic_df.loc[anomaly_indices, "calls_abandoned"] * np.random.uniform(2.0, 4.0, anomaly_count)
)
synthetic_df.loc[anomaly_indices, "calls_answered"] = (
    synthetic_df.loc[anomaly_indices, "calls_answered"] * np.random.uniform(0.6, 0.9, anomaly_count)
)

# keep values logically valid again
synthetic_df["calls_answered"] = np.minimum(synthetic_df["calls_answered"], synthetic_df["calls_offered"])
synthetic_df["calls_abandoned"] = np.minimum(synthetic_df["calls_abandoned"], synthetic_df["calls_offered"])

# recalculate features after anomaly injection
synthetic_df["unanswered_calls"] = synthetic_df["calls_offered"] - synthetic_df["calls_answered"]
synthetic_df["answer_rate"] = np.where(
    synthetic_df["calls_offered"] > 0,
    synthetic_df["calls_answered"] / synthetic_df["calls_offered"],
    0
)
synthetic_df["abandon_rate"] = np.where(
    synthetic_df["calls_offered"] > 0,
    synthetic_df["calls_abandoned"] / synthetic_df["calls_offered"],
    0
)
synthetic_df["calls_per_1000_population"] = np.where(
    synthetic_df["population"] > 0,
    (synthetic_df["calls_offered"] / synthetic_df["population"]) * 1000,
    0
)

synthetic_df.loc[anomaly_indices, "is_anomaly"] = 1

print("Synthetic dataset shape:", synthetic_df.shape)
print(synthetic_df["is_anomaly"].value_counts())
synthetic_df.head()

## 10. Save the synthetic labelled dataset

In [ ]:
synthetic_output_path = "nhs_111_synthetic_anomaly_dataset.csv"
synthetic_df.to_csv(synthetic_output_path, index=False)
synthetic_output_path

## 11. Train an Isolation Forest model

We use only numeric service features for anomaly detection.

In [ ]:
feature_columns = [
    "population",
    "calls_offered",
    "calls_abandoned",
    "calls_answered",
    "unanswered_calls",
    "answer_rate",
    "abandon_rate",
    "calls_per_1000_population"
]

X = synthetic_df[feature_columns].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)

synthetic_df["predicted_anomaly_raw"] = model.fit_predict(X_scaled)
synthetic_df["predicted_anomaly"] = synthetic_df["predicted_anomaly_raw"].map({1: 0, -1: 1})

synthetic_df[["is_anomaly", "predicted_anomaly"]].head()

## 12. Evaluate the model against the synthetic labels

In [ ]:
print(classification_report(synthetic_df["is_anomaly"], synthetic_df["predicted_anomaly"]))

## 13. Review the most unusual records

In [ ]:
synthetic_df["anomaly_score"] = model.decision_function(X_scaled)

top_anomalies = synthetic_df.sort_values("anomaly_score").head(15)
top_anomalies[[
    "region",
    "provider",
    "period",
    "calls_offered",
    "calls_answered",
    "calls_abandoned",
    "answer_rate",
    "abandon_rate",
    "is_anomaly",
    "predicted_anomaly",
    "anomaly_score"
]]

## 14. Visualise anomalies by call volume

In [ ]:
plot_df = synthetic_df.sort_values("period").reset_index(drop=True)

plt.plot(plot_df.index, plot_df["calls_offered"], label="Calls offered")
anomaly_points = plot_df[plot_df["predicted_anomaly"] == 1]

plt.scatter(anomaly_points.index, anomaly_points["calls_offered"], label="Predicted anomaly")
plt.title("Calls offered with predicted anomalies")
plt.xlabel("Record index")
plt.ylabel("Calls offered")
plt.legend()
plt.show()

## 15. Visualise answer rate and abandon rate

In [ ]:
plt.plot(plot_df.index, plot_df["answer_rate"], label="Answer rate")
plt.scatter(anomaly_points.index, anomaly_points["answer_rate"], label="Predicted anomaly")
plt.title("Answer rate with predicted anomalies")
plt.xlabel("Record index")
plt.ylabel("Answer rate")
plt.legend()
plt.show()

plt.plot(plot_df.index, plot_df["abandon_rate"], label="Abandon rate")
plt.scatter(anomaly_points.index, anomaly_points["abandon_rate"], label="Predicted anomaly")
plt.title("Abandon rate with predicted anomalies")
plt.xlabel("Record index")
plt.ylabel("Abandon rate")
plt.legend()
plt.show()

## 16. Region level summary

In [ ]:
region_summary = (
    synthetic_df.groupby("region")
    .agg(
        total_records=("region", "count"),
        true_anomalies=("is_anomaly", "sum"),
        predicted_anomalies=("predicted_anomaly", "sum"),
        avg_calls_offered=("calls_offered", "mean"),
        avg_answer_rate=("answer_rate", "mean"),
        avg_abandon_rate=("abandon_rate", "mean")
    )
    .reset_index()
    .sort_values("predicted_anomalies", ascending=False)
)

region_summary.head(10)

## 17. Key takeaway and conclusion

### Key takeaways

- The NHS 111 Excel workbook can be transformed into a usable analytical dataset by rebuilding the two row header and selecting the operational service fields.
- The real dataset is suitable for **service level anomaly detection** such as unusual demand spikes, low answer rates, and high abandon rates.
- Because the source is aggregated and unlabelled, a **synthetic labelled dataset** is a practical and realistic way to prototype anomaly detection models.
- Isolation Forest works well as a first unsupervised model for finding unusual operational patterns.
- Useful features for this problem include:
  - calls offered
  - calls answered
  - calls abandoned
  - unanswered calls
  - answer rate
  - abandon rate
  - calls per 1000 population

### Conclusion

This notebook shows a realistic healthcare analytics workflow:

**real NHS open data → cleaning → feature engineering → synthetic label generation → anomaly detection → interpretation**

This is a strong foundation for:
- Jupyter notebook assignments  
- Gradio demos  
- FastAPI model serving  
- NHS style operational monitoring projects  

For production use, the next step would be to add:
- time based baseline comparison  
- rule based alerts  
- dashboard monitoring  
- governance and explanation for operational decisions